# ⚠️ 실습 전 필수: 본인 드라이브에 사본 만들기!

상단 메뉴 **파일 → 드라이브에 사본 저장**을 클릭하세요.

사본을 만들지 않으면 작성한 코드가 저장되지 않습니다. 꼭 먼저 사본을 만든 뒤 시작하세요!

## 2주차 · LLM API 와 Agent 기초 (Responses API / Chat Completions)

지난 주에는 PyTorch로 딥러닝의 기본기를 다졌습니다. 이번 주에는 이미 학습이 끝난 거대 언어 모델(LLM)을
**API로 불러다 쓰는 법**과, 모델이 스스로 도구(tool)를 호출하며 문제를 풀어가는 **에이전트(Agent)의 기초**를 배웁니다.

강의의 주제는 OpenAI의 [**Responses API**](https://platform.openai.com/docs/api-reference/responses) 입니다.
다만 Responses API는 유료 OpenAI 키가 필요하므로, 이 실습에서는 **무료로 실행 가능한 환경**을 함께 사용합니다.

- 🧪 **직접 실행하는 실습** → [**OpenRouter**](https://openrouter.ai) 의 무료 모델 `google/gemma-4-31b-it:free` 를 **Chat Completions API**로 사용합니다. (무료, 카드 등록 불필요)
- 📖 **개념 비교** → 같은 기능을 OpenAI **Responses API**로는 어떻게 쓰는지 코드로 나란히 보여줍니다.

두 API는 겉모습(입출력 형태)만 조금 다를 뿐, **function calling·에이전트의 핵심 개념은 동일**합니다.

### 오늘의 목표
1. OpenRouter 가입 & 무료 API 키 발급
2. LLM에 첫 요청 보내고 응답 받기 (Chat Completions)
3. Responses API와 Chat Completions의 차이 이해하기
4. 대화 맥락 이어가기
5. 모델에게 **함수(도구)** 를 쥐여주고 호출하게 만들기 (function calling)
6. 함수 실행 결과를 되돌려주는 **에이전트 루프** 직접 구현하기

---
## 1. OpenRouter 가입 & API 키 발급

[**OpenRouter**](https://openrouter.ai) 는 하나의 API로 여러 회사의 LLM(OpenAI·구글·메타 등)을 골라 쓸 수 있게 해주는 중계 서비스입니다.
일부 모델은 **무료**로 제공되어, 이 실습처럼 학습·테스트 용도로 쓰기 좋습니다.

### 가입 및 키 발급 절차

1. [**openrouter.ai**](https://openrouter.ai) 접속 → 우측 상단 **Sign In** 클릭
2. **Google** 또는 **GitHub** 계정으로 로그인 (별도 회원가입 없이 소셜 로그인 가능)
3. 로그인 후 [**openrouter.ai/keys**](https://openrouter.ai/keys) 로 이동 (또는 우측 상단 프로필 → **Keys**)
4. **Create Key** 버튼 클릭 → 키 이름(예: `week2-practice`) 입력 → **Create**
5. 생성된 키(`sk-or-v1-...`)를 **복사**해 둡니다. ⚠️ 이 화면을 벗어나면 다시 볼 수 없으니 지금 복사하세요.

> 💳 **무료 모델과 한도:** `:free` 가 붙은 모델은 크레딧(결제) 없이 사용할 수 있습니다.
> 대신 사용량 제한이 있습니다 — 이 실습 모델 기준 대략 **분당 20회 / 하루 200회** 정도이며, 정책은 바뀔 수 있으니
> 정확한 한도는 [openrouter.ai](https://openrouter.ai/google/gemma-4-31b-it:free)에서 확인하세요. 실습에는 충분합니다.

---
## 2. 환경 설정

OpenRouter는 **OpenAI 파이썬 라이브러리와 호환**됩니다. `base_url`만 OpenRouter 주소로 바꿔주면
평소 OpenAI를 쓰듯 그대로 사용할 수 있습니다. 먼저 라이브러리를 설치합니다.

In [ ]:
!pip install -q --upgrade openai

### API 키 입력

방금 발급받은 OpenRouter 키를 입력합니다.

> 🔐 **주의:** API 키는 비밀번호와 같습니다. 코드에 직접 붙여넣어 공유하지 마세요.
> 아래 셀은 입력창에 키를 붙여넣어도 화면에 노출되지 않도록 `getpass`를 사용합니다.

In [ ]:
import os
from getpass import getpass

# Colab 왼쪽 🔑(보안 비밀) 메뉴에 OPENROUTER_API_KEY 를 저장해 두었다면 그 값을 쓰고,
# 없으면 직접 입력받습니다.
if not os.environ.get("OPENROUTER_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    except Exception:
        os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API 키(sk-or-v1-...)를 입력하세요: ")

print("API 키 설정 완료 ✅")

In [ ]:
from openai import OpenAI

# OpenAI 라이브러리를 그대로 쓰되, 접속 주소만 OpenRouter 로 바꿉니다.
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

# 앞으로 계속 쓸 모델. OpenRouter의 무료 모델입니다.
# 만약 RateLimitError: Error code: 429가 뜬다면 "openai/gpt-oss-20b:free"로 바꿔서 재시도 해주세요. 
MODEL = "google/gemma-4-31b-it:free"

---
## 3. 첫 번째 응답 받기 (Chat Completions)

가장 기본형은 `client.chat.completions.create(...)` 입니다.
대화를 `messages` 리스트로 넣고, 결과는 `response.choices[0].message.content` 로 꺼냅니다.

`messages`의 각 항목은 `role`(누가 말했나)과 `content`(내용)를 가집니다.
- `"user"` — 사용자(나)
- `"assistant"` — 모델
- `"system"` — 모델의 역할·규칙을 정하는 지시문

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
         #  딥러닝이 뭔지 초등학생도 이해할 수 있게 두 문장으로 설명해줘 
        {"role": "user", "content": "자연어 처리에 대해 전혀 모르는 사람도 이해할 수 있게 두 문장으로 설명해줘."},
    ],
)

print(response.choices[0].message.content)

`response` 객체에는 답변 텍스트 외에 사용 토큰량 등의 정보도 들어 있습니다.

In [ ]:
print("사용 모델:", response.model)
print("토큰 사용량:", response.usage)

---
## 4. 📖 [참고] OpenAI Responses API 는 무엇이 다른가

강의의 주제인 **Responses API**는 OpenAI가 내놓은 최신 인터페이스로, 위의 Chat Completions보다 더 단순하게 설계되었습니다.
OpenAI 유료 키가 있다면 아래처럼 씁니다. **(OpenRouter에서는 지원하지 않아 이 실습에서는 실행하지 않습니다. 개념만 익혀두세요.)**

```python
from openai import OpenAI
client = OpenAI(api_key="OpenAI 키")          # base_url 없음 = OpenAI 본사

# 입력은 input=, 출력은 output_text 로 아주 간단
resp = client.responses.create(
    model="gpt-4.1",
    input="딥러닝을 두 문장으로 설명해줘.",
)
print(resp.output_text)

# 대화 이어가기도 이전 응답 id 한 줄이면 끝 (상태를 서버가 기억)
resp2 = client.responses.create(
    model="gpt-4.1",
    input="방금 설명을 한 문장으로 줄여줘.",
    previous_response_id=resp.id,             # 👈 Chat Completions에는 없는 기능
)
print(resp2.output_text)
```

### 한눈에 보는 차이

| | **Chat Completions** (이번 실습, OpenRouter) | **Responses API** (OpenAI) |
|---|---|---|
| 호출 | `client.chat.completions.create` | `client.responses.create` |
| 입력 | `messages=[{"role":..., "content":...}]` | `input="..."` |
| 출력 | `response.choices[0].message.content` | `response.output_text` |
| 대화 유지 | `messages` 리스트를 **직접** 관리 | `previous_response_id` (서버가 기억) |
| 함수 스키마 | `{"type":"function", "function":{...}}` (한 겹 감쌈) | `{"type":"function", ...}` (평평) |

👉 형태만 다를 뿐, "모델에 요청하고 답을 받는다 / 도구를 호출한다"는 **핵심은 똑같습니다.**
아래부터는 다시 무료로 실행 가능한 **Chat Completions(OpenRouter)** 로 실습을 이어갑니다.

---
## 5. system 메시지로 역할(페르소나) 정하기

`system` 역할 메시지는 모델에게 "너는 이런 존재야"라고 알려주는 지시문입니다.
(Responses API의 `instructions`에 해당합니다.) 같은 질문이라도 system에 따라 말투와 관점이 달라집니다.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "너는 30년 경력의 친절한 국어 선생님이야. 항상 존댓말을 쓰고, 어려운 말은 쉽게 풀어서 설명해."},
        {"role": "user", "content": "'벡터'라는 단어의 뜻을 알려줘."},
    ],
)

print(response.choices[0].message.content)

### 🔧 파라미터로 답변 조절하기 — `temperature` (온도)

`temperature`는 답변의 **무작위성**을 조절하는 값입니다. **0 ~ 2** 사이로 설정하며(주로 **0.0 ~ 1.0** 사용),
값이 **높을수록 창의적이고 다양한** 응답을, **낮을수록 정확하고 일관된** 응답을 생성합니다.
낮은 온도에서는 모델이 "가장 그럴듯한 단어"만 고르고, 온도가 높아질수록 덜 흔한 단어까지 선택 후보에 넣습니다.

**주요 설정 범위 및 활용 사례**

| 범위 | 성격 | 특징 | 활용 사례 |
|---|---|---|---|
| **0.0 ~ 0.3** | 저온 / 사실 기반 | 가장 가능성이 높은 단어만 선택 → 높은 정확도·일관성 | 수학, 코딩, 사실 기반 요약, 데이터 추출 |
| **0.4 ~ 0.7** | 중간 / 균형 | 예측 가능성과 창의성의 균형 | 일반 대화, 정보 검색 (이상적인 출발점) |
| **0.7 ~ 1.0** | 고온 / 창의적 | 덜 일반적인 단어의 선택 확률↑ | 브레인스토밍, 소설 작성, 마케팅 문구 |
| **1.0 이상** | 매우 높음 / 실험적 | 매우 무작위 → 엉뚱하거나 문맥에 안 맞는 결과 가능 | 실험적 용도 |

- `max_tokens` — 응답 길이(생성할 토큰 수)의 상한.

아래 셀의 `temperature` 를 **0.0 → 0.5 → 1.5** 로 바꿔가며 같은 프롬프트를 여러 번 실행하고,
답이 얼마나 달라지는지(0.0에서는 거의 매번 같은 답, 높을수록 매번 다른 답) 직접 비교해 보세요.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "'가을'을 주제로 한 줄짜리 시를 지어줘."}],
    temperature=1.2,
    max_tokens=100,
)

print(response.choices[0].message.content)

---
## 6. 대화 이어가기 — `messages` 직접 관리

Chat Completions는 상태를 서버가 기억하지 않습니다. 그래서 대화를 이어가려면
**지금까지의 모든 메시지를 리스트에 쌓아** 매번 함께 보내야 합니다.
(Responses API였다면 `previous_response_id` 한 줄로 끝나는 부분입니다.)

In [ ]:
# 대화 기록을 담을 리스트
messages = [
    {"role": "user", "content": "내가 좋아하는 숫자는 7이야. 기억해둬."},
]

first = client.chat.completions.create(model=MODEL, messages=messages)
answer1 = first.choices[0].message.content
print("1차 응답:", answer1)

# 모델의 답변도 기록에 추가해야 다음 턴에서 맥락이 유지됩니다.
messages.append({"role": "assistant", "content": answer1})

In [ ]:
# 이어지는 질문을 기록에 추가하고 다시 전체를 보냄
messages.append({"role": "user", "content": "내가 좋아하는 숫자에 3을 곱하면 얼마야?"})

second = client.chat.completions.create(model=MODEL, messages=messages)
print("2차 응답:", second.choices[0].message.content)

`messages`에 이전 대화를 넣지 않으면 모델은 "내가 좋아하는 숫자"가 뭔지 알지 못합니다.
직접 리스트에서 빼보고 답이 어떻게 달라지는지 확인해 보세요.

---
## 7. 함수 호출(Function Calling) 기초

LLM은 최신 정보나 실시간 계산(예: 오늘 날씨, 환율, 데이터베이스 조회)은 스스로 하지 못합니다.
대신 우리가 **함수(도구)** 를 정의해 모델에게 알려주면, 모델은 필요할 때
"이 함수를 이런 인자로 실행해줘"라고 **요청**합니다. 실제 실행은 우리 코드가 합니다.

먼저 도구로 쓸 파이썬 함수를 하나 만듭니다. (실습이므로 실제 날씨 대신 가짜 데이터를 반환합니다.)

In [ ]:
def get_weather(city: str) -> str:
    # 주어진 도시의 (가짜) 현재 날씨를 반환한다.
    fake_db = {
        "서울": "맑음, 27도",
        "부산": "흐림, 24도",
        "제주": "비, 22도",
    }
    return fake_db.get(city, f"{city}의 날씨 정보가 없습니다.")

# 잘 동작하는지 먼저 확인
print(get_weather("서울"))
print(get_weather("도쿄"))

### 도구 스키마 정의

모델에게 이 함수의 **이름, 설명, 필요한 인자**를 JSON 스키마로 알려줍니다.
Chat Completions에서는 함수 정보를 `"function"` 키로 한 번 감싸는 형태를 씁니다.
(앞의 비교표에서 봤듯 Responses API는 이 감싸는 층이 없습니다.)

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "특정 도시의 현재 날씨를 알려준다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "날씨를 조회할 도시 이름. 예: 서울, 부산",
                    },
                },
                "required": ["city"],
            },
        },
    }
]

이제 도구를 함께 넘겨 모델에게 질문합니다.
모델이 "함수를 호출해야겠다"고 판단하면, 답변 텍스트(`content`) 대신
**함수 호출 요청**을 `message.tool_calls` 에 담아 돌려줍니다.

In [ ]:
messages = [{"role": "user", "content": "서울 날씨 어때?"}]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
)

message = response.choices[0].message
print("일반 답변 content:", message.content)   # 함수를 호출하려 할 땐 보통 None
print("함수 호출 요청 tool_calls:")
for call in (message.tool_calls or []):
    print("  호출할 함수:", call.function.name)
    print("  인자(문자열):", call.function.arguments)
    print("  call id:", call.id)

---
## 8. 함수 실행 결과를 모델에게 돌려주기

모델은 "`get_weather`를 `city=서울`로 실행해줘"라고 요청했을 뿐, 아직 최종 답은 하지 않았습니다.
이제 우리가 할 일은 세 가지입니다.

1. 모델이 요청한 함수를 **실제로 실행**한다.
2. 실행 결과를 `role: "tool"` 메시지로 만들어 대화에 이어 붙인다. (`tool_call_id`로 어느 요청의 결과인지 연결)
3. 이 대화를 모델에게 **다시** 보내 최종 답변을 받는다.

In [ ]:
import json

# 1) 모델의 함수 호출 요청 메시지 자체를 먼저 기록에 추가합니다.
messages.append(message)

# 2) 요청된 함수를 실행하고, 결과를 tool 메시지로 붙입니다.
for call in message.tool_calls:
    args = json.loads(call.function.arguments)      # '{"city": "서울"}' -> dict
    result = get_weather(**args)                    # 실제 파이썬 함수 실행
    messages.append({
        "role": "tool",
        "tool_call_id": call.id,                    # 어떤 호출에 대한 결과인지 연결
        "content": result,
    })

# 3) 결과를 담아 다시 모델에게 보냄 -> 자연어 최종 답변
final = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
)
print(final.choices[0].message.content)

---
## 9. 에이전트(Agent) 루프 만들기

위 과정을 정리하면 이렇습니다.

```
사용자 질문 → 모델 → (함수 호출 요청?) → 함수 실행 → 결과 전달 → 모델 → ...
```

모델은 필요하면 함수를 **여러 번, 여러 종류** 호출할 수 있습니다.
그래서 "함수 호출이 없을 때까지 반복"하는 루프로 감싸면, 모델이 알아서 도구를 골라 쓰며
문제를 풀어가는 **에이전트**가 됩니다.

도구를 하나 더 추가해 봅시다 — 간단한 계산기입니다.

In [ ]:
def calculate(expression: str) -> str:
    # 간단한 사칙연산 문자열을 계산한다. 예: '7 * 3'
    # 실습용 최소 구현. 실제 서비스에서는 eval 대신 안전한 파서를 써야 합니다.
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return "허용되지 않은 문자가 있습니다."
    try:
        return str(eval(expression))
    except Exception as e:
        return f"계산 오류: {e}"

# 이름 -> 실제 함수 로 연결하는 표(dispatch table)
available_functions = {
    "get_weather": get_weather,
    "calculate": calculate,
}

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "특정 도시의 현재 날씨를 알려준다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "도시 이름. 예: 서울"},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "사칙연산 수식을 계산한다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "계산할 수식. 예: '12 * 8'"},
                },
                "required": ["expression"],
            },
        },
    },
]

In [ ]:
import json

def run_agent(user_message, max_turns=5):
    # 모델이 함수 호출을 멈출 때까지 반복하는 최소 에이전트 루프.
    messages = [{"role": "user", "content": user_message}]

    for turn in range(max_turns):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
        )
        message = response.choices[0].message
        messages.append(message)   # 모델의 응답(함수 호출 포함)을 대화에 누적

        # 함수 호출 요청이 없으면 최종 답변이 나온 것 -> 종료
        if not message.tool_calls:
            return message.content

        # 요청된 함수들을 실행해 결과를 대화에 붙임
        for call in message.tool_calls:
            func = available_functions[call.function.name]
            args = json.loads(call.function.arguments)
            result = func(**args)
            print(f"  🔧 {call.function.name}({args}) -> {result}")
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": str(result),
            })

    return "(최대 반복 횟수를 초과했습니다.)"

이제 에이전트를 호출해 봅시다. 아래 질문은 **날씨 조회와 계산을 둘 다** 해야 답할 수 있습니다.
모델이 도구를 어떤 순서로 쓰는지 `🔧` 로그로 지켜보세요.

> ⚠️ 무료 모델은 성능이 제한적이라, 도구를 한 번에 완벽히 쓰지 못할 때도 있습니다.
> 잘 안 되면 질문을 더 명확히 하거나 여러 번 실행해 보세요.

In [ ]:
answer = run_agent("서울과 부산 날씨를 알려주고, 25 곱하기 4가 얼마인지도 계산해줘.")
print("\n최종 답변:\n", answer)

---
## 10. 연습문제 🎯

지금까지 에이전트가 쓰던 `get_weather` 는 미리 적어둔 **가짜 날씨**를 돌려주는 도구였습니다.
실제로 밖에 비가 와도 서울은 언제나 "맑음, 27도"였죠.

이번에는 이 도구를 **진짜 날씨 API**로 업그레이드해서,
에이전트가 지금 이 순간의 실제 데이터로 답하게 만들어 봅니다.

날씨 API는 [**Open-Meteo**](https://open-meteo.com) 를 사용합니다.
**가입도 API 키도 필요 없는** 무료 서비스라 바로 실습할 수 있습니다.

`# TODO` 로 표시된 부분만 하나씩 채우면 됩니다. 천천히 따라오세요!

### 문제 1. 진짜 날씨 도구 `get_real_weather` 를 에이전트에 추가하기

새 도구를 붙이는 순서는 7장에서와 언제나 같습니다.

**① 함수 만들기 → ② 이름표에 등록 → ③ 스키마 추가 → ④ 테스트**

<br>

**1단계. 함수 만들기**

아래 함수는 두 번의 API 호출로 실제 날씨를 가져옵니다.

1. 도시 이름으로 위도·경도를 찾고 (지오코딩)
2. 그 좌표의 현재 날씨를 조회합니다.

API 호출 부분은 미리 작성해 두었습니다.
여러분이 채울 곳은 마지막 `return` 한 줄입니다.

> ⭐ 도구가 반환한 문자열은 `role:"tool"` 메시지로 전달되고, 모델은 **그 문자열만 보고** 답을 만듭니다.
> `"23"` 만 주면 기온인지 습도인지 알 수 없으니,
> `"서울: 기온 23°C, 습도 60%"` 처럼 **이름표와 단위를 붙인 문장**으로 반환하세요.

In [ ]:
import requests

def get_real_weather(city: str) -> str:
    # Open-Meteo API로 도시의 '진짜' 현재 날씨를 조회한다. (무료, API 키 불필요)
    try:
        # 1) 도시 이름 -> 위도/경도 (한국어 이름도 가능)
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1, "language": "ko"},
            timeout=5,
        ).json()
        if not geo.get("results"):
            return f"'{city}' 도시를 찾을 수 없습니다."
        loc = geo["results"][0]

        # 2) 위도/경도 -> 현재 날씨
        w = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": loc["latitude"],
                "longitude": loc["longitude"],
                "current": "temperature_2m,apparent_temperature,relative_humidity_2m,wind_speed_10m",
            },
            timeout=5,
        ).json()["current"]
    except requests.exceptions.RequestException as e:
        return f"네트워크 오류: {e}"

    # 사용 가능한 값:
    #   loc["name"]                -> 도시 이름 (예: "서울")
    #   w["temperature_2m"]        -> 기온 (°C)
    #   w["apparent_temperature"]  -> 체감 온도 (°C)
    #   w["relative_humidity_2m"]  -> 습도 (%)
    #   w["wind_speed_10m"]        -> 풍속 (km/h)

    # TODO: 위 값들을 이름표·단위가 붙은 한 문장으로 만들어 반환하세요.
    #   힌트: return f"{loc['name']}: 기온 {w['temperature_2m']}°C (체감 ...), 습도 ..., 풍속 ..."
    return  # <- 여기를 채우세요

# 진짜 오늘 날씨가 나오는지 확인해 보세요.
print(get_real_weather("서울"))
print(get_real_weather("Busan"))
print(get_real_weather("없는도시123"))   # 오류 처리 확인

**2단계. 함수를 이름표에 등록**

모델은 함수를 직접 실행하지 못합니다.
"`get_real_weather` 를 실행해줘"라는 **이름(문자열)**을 보낼 뿐이죠.

그 이름으로 진짜 파이썬 함수를 찾아 실행하는 것이 `available_functions` 딕셔너리입니다.

새 도구를 등록하세요. 가짜 `get_weather` 는 헷갈리지 않게 치워 둡시다.

In [ ]:
# TODO: "get_real_weather" 라는 이름으로 get_real_weather 함수를 등록하세요.
#   힌트: available_functions["이름"] = 함수


# 가짜 날씨 도구 제거 (그대로 실행하면 됩니다)
available_functions.pop("get_weather", None)
tools[:] = [t for t in tools if t["function"]["name"] != "get_weather"]

# dict_keys(['calculate', 'get_real_weather']) 가 보여야 합니다.
print(available_functions.keys())

**3단계. 도구 스키마 추가**

모델은 우리가 짠 파이썬 코드를 보지 못합니다.
**스키마에 적은 설명만 읽고** 도구를 쓸지, 인자를 뭐라고 채울지 결정합니다.

채울 곳은 세 군데입니다.

| 채울 곳 | 역할 |
|---|---|
| `name` | 2단계 등록 이름과 **정확히** 동일해야 함 (다르면 `KeyError`) |
| `description` | 모델이 이 도구를 **언제** 쓸지 판단하는 기준 |
| `city` 의 `description` | 인자를 **어떻게** 채울지 안내 (예시 값을 함께 적기) |

In [ ]:
tools.append({
    "type": "function",
    "function": {
        "name": "",         # TODO ①: 2단계에서 등록한 이름과 똑같이
        "description": "",  # TODO ②: 예: "특정 도시의 실제 현재 날씨(기온·습도·풍속)를 조회한다."
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "",  # TODO ③: 예: "날씨를 조회할 도시 이름. 예: 서울, Busan"
                },
            },
            "required": ["city"],
        },
    },
})

# ['calculate', 'get_real_weather'] 가 보여야 합니다.
print([t["function"]["name"] for t in tools])

**4단계. 테스트!**

9장의 `run_agent` 는 `tools` 와 `available_functions` 를 그대로 쓰므로,
방금 추가한 새 도구를 바로 사용할 수 있습니다.

아래 질문은 날씨 조회와 계산을 **둘 다** 해야 답할 수 있습니다.
🔧 로그에 `get_real_weather` 가 찍히면 성공입니다. 창밖과 비교해 보세요! ☀️

In [ ]:
print(run_agent("서울의 지금 실제 날씨를 알려주고, 현재 기온에 2를 곱하면 얼마인지도 계산해줘."))

### 문제 2. 에이전트 루프 복원하기

9장에서 만든 에이전트 루프의 흐름입니다.

```
질문 → 모델 → 도구 호출 요청? ── 없음 → ① 최종 답변, 종료
                │ 있음
                ▼
        ② 이름으로 진짜 함수를 찾아 실행
                ▼
        ③ 결과를 모델에게 반환 → 다시 모델에게
```

이번에는 이 루프를 직접 완성합니다.
그림의 ①②③ 세 곳이 빈칸(`________`)으로 비어 있습니다.

9장 코드를 그대로 베끼기보다, 각 빈칸이 **왜 필요한지** 생각하며 채워 보세요.

> ⚠️ 빈칸을 채우기 전에 실행하면 `NameError` 가 납니다. 정상이니 당황하지 마세요!

In [ ]:
import json

def run_agent_v2(user_message, system_prompt="", max_turns=5):
    messages = []
    if system_prompt:   # system 프롬프트가 있으면 맨 앞에 (문제 3에서 사용)
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_message})

    for turn in range(max_turns):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        message = response.choices[0].message
        messages.append(message)

        # TODO ①: 도구 호출 요청이 없으면 최종 답변. (힌트: message 의 어떤 속성이었나요?)
        if not ________:
            return message.content

        for call in message.tool_calls:
            # TODO ②: 모델이 보낸 이름으로 진짜 함수 찾기. (힌트: call.function.____)
            func = available_functions[________]
            args = json.loads(call.function.arguments)
            result = func(**args)
            print(f"  🔧 {call.function.name}({args}) -> {result}")

            # TODO ③: 결과를 모델에게 돌려주는 메시지입니다. role 과 tool_call_id 를 채우세요. (8장 참고)
            messages.append({
                "role": ________,
                "tool_call_id": ________,
                "content": str(result),
            })

    return "(최대 반복 횟수를 초과했습니다.)"

# 이 질문은 도구를 두 번 호출해야 답할 수 있습니다. 🔧 로그가 두 줄 찍히는지 관찰하세요.
print(run_agent_v2("서울이랑 부산 중에 지금 어디가 더 따뜻해?"))

### 문제 3. 기상 캐스터 페르소나 만들기

마지막 빈칸은 코드가 아니라 **프롬프트**입니다.

5장에서 배운 `system` 메시지는 말투뿐 아니라 에이전트의 **행동 규칙**을 정하는 곳이기도 합니다.
특히 조건 ③ "도구로 확인한 데이터만 말하기"는
에이전트가 정보를 지어내는 것을 막기 위해 실제 서비스에서도 널리 쓰이는 방법입니다.

In [ ]:
# TODO: 아래 세 조건을 모두 담은 system 프롬프트를 작성하세요.
#   ① 모든 답변을 '📺 날씨 캐스터입니다.' 로 시작하기
#   ② 날씨에 맞는 옷차림 조언을 한 문장 덧붙이기
#   ③ 도구로 조회한 실제 데이터만 근거로 말하고, 조회하지 않은 값은 지어내지 않기
weather_caster_prompt = ""   # <- 여기를 채우세요

# 답이 '📺 날씨 캐스터입니다.' 로 시작하고, 실제 날씨 + 옷차림 조언이 나오면 성공!
print(run_agent_v2("서울 날씨 어때? 오늘 뭐 입고 나갈까?", system_prompt=weather_caster_prompt))

---
## 정리

이번 주에 배운 것:

- **OpenRouter** 무료 모델로 LLM API 실습 환경 만들기
- **Chat Completions** 기본 사용법 — `messages`, `choices[0].message.content`
- **Responses API** 와의 차이 — `input`/`output_text`, `previous_response_id` (개념 비교)
- **system 메시지 / temperature** 로 역할과 답변 성향 조절
- **messages 직접 관리** 로 대화 맥락 이어가기
- **Function calling** — 도구 스키마 정의 → 모델의 호출 요청(`tool_calls`) → 실행 → `role:"tool"` 로 결과 전달
- **에이전트 루프** — 함수 호출이 없어질 때까지 반복하며 모델이 스스로 도구를 사용하게 하기
- **외부 API 연동** — Open-Meteo 실시간 날씨를 도구로 연결하고, 좋은 반환값·스키마 설명·행동 규칙이 에이전트 품질을 결정한다는 것

다음 단계로는 웹 검색·코드 실행 같은 **내장 도구(built-in tools)** 와, 여러 에이전트를 엮는
**Agents SDK** 로 확장해 볼 수 있습니다. 수고하셨습니다! 🎉